# Study 04 ENNI Context Ablation

This notebook runs the full `prev_same_speaker` ENNI context-window experiment from this repo and compares it against the existing `utterance_only` outputs.

Workflow:
1. Clone your repo branch.
2. Install inference dependencies.
3. Set your Hugging Face token.
4. Build the ENNI evaluation JSONL from the clean `.cha` files.
5. Reuse the existing `utterance_only` predictions.
6. Run the model once in `prev_same_speaker` mode on the full reviewed ENNI set.
7. Compare the outputs on shared `row_id`s.


In [ ]:
REPO_URL = "https://github.com/shamira-venturini/talkbank-morphosyntax-error-annotator.git"
BRANCH = "<YOUR-BRANCH>"
REPO_DIR = "/content/talkbank-morphosyntax-error-annotator"

# Full run.
LIMIT = 0
BATCH_SIZE = 4

# Must point to the full clean ENNI .cha tree. If the transcripts are only in Drive,
# replace this with the mounted Drive path before running.
CLEAN_ENNI_DIR = "study_04_context_windows/ENNI"
FULL_INPUT_JSONL = "study_04_context_windows/data/context_eval/enni_reviewed_prev_same_speaker_eval_full.jsonl"
FULL_SUMMARY_JSON = "study_04_context_windows/data/context_eval/enni_reviewed_prev_same_speaker_eval_full_summary.json"
STAGE3_SPLIT = "study_01_talkbank_tool_paper/experiments/recon_full_comp_preserve/stage3_train.jsonl"
CHAT_TOKENS = "study_01_talkbank_tool_paper/experiments/recon_full_comp_preserve/chat_tokens.json"
PREV_OUTPUT_DIR = "study_04_context_windows/results/enni_context_ablation_full_prev_same_speaker"

# Existing utterance_only baseline. The analysis script falls back to structural
# alignment when row_ids differ across files.
EXISTING_UTTERANCE_ONLY_JSONL = "study_02_hitl_adaptation/results/OOD_eval/enni_ood_chat/predictions_utterance_only.jsonl"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git checkout "$BRANCH"
!git status --short


In [ ]:
!nvidia-smi
!python3 --version


In [ ]:
!pip install -q transformers peft bitsandbytes accelerate sentencepiece scipy orjson


In [ ]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")


In [ ]:
%cd "$REPO_DIR"
from pathlib import Path

repo_dir = Path(REPO_DIR)
candidate_enni_dirs = [
    repo_dir / CLEAN_ENNI_DIR,
    Path(CLEAN_ENNI_DIR),
    Path("/content/drive/MyDrive/ENNI"),
    Path("/content/drive/MyDrive/study_04_context_windows/ENNI"),
]
resolved_clean_enni_dir = next((path for path in candidate_enni_dirs if path.exists()), None)
if resolved_clean_enni_dir is None:
    raise FileNotFoundError(
        "Clean ENNI transcript tree not found. Set CLEAN_ENNI_DIR to the full .cha directory in Drive or add the transcripts to the repo clone."
    )
cha_count = len(list(resolved_clean_enni_dir.rglob("*.cha")))
if cha_count == 0:
    raise FileNotFoundError(f"No .cha files found under {resolved_clean_enni_dir}")
if resolved_clean_enni_dir.is_relative_to(repo_dir):
    CLEAN_ENNI_DIR = str(resolved_clean_enni_dir.relative_to(repo_dir))
else:
    CLEAN_ENNI_DIR = str(resolved_clean_enni_dir)
print("Using CLEAN_ENNI_DIR:", CLEAN_ENNI_DIR)
print("CHA files:", cha_count)

!python3 study_04_context_windows/scripts/prepare_enni_context_ablation_input.py \
  --enni-dir "$CLEAN_ENNI_DIR" \
  --out-jsonl "$FULL_INPUT_JSONL" \
  --out-summary "$FULL_SUMMARY_JSON" \
  --limit "$LIMIT"


Validate the existing `utterance_only` prediction file. This notebook does not rerun that condition.


In [ ]:
import json
from pathlib import Path

utterance_only_path = Path(REPO_DIR) / EXISTING_UTTERANCE_ONLY_JSONL
full_summary_path = Path(REPO_DIR) / FULL_SUMMARY_JSON

print("utterance_only baseline:", utterance_only_path)
print("exists:", utterance_only_path.exists())
print("full prep summary:", full_summary_path)
summary = json.loads(full_summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))
if summary["rows_exported"] == 0:
    raise RuntimeError("Prep exported 0 rows. Check CLEAN_ENNI_DIR and the unresolved_examples in the summary.")


Run the frozen model once on the full ENNI JSONL in `prev_same_speaker` mode.


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/run_ood_context_inference.py \
  --input-jsonl "$FULL_INPUT_JSONL" \
  --context-mode prev_same_speaker \
  --stage3-split "$STAGE3_SPLIT" \
  --chat-tokens "$CHAT_TOKENS" \
  --out-dir "$PREV_OUTPUT_DIR" \
  --batch-size "$BATCH_SIZE" \
  --limit "$LIMIT"


Compare the existing `utterance_only` predictions against the new `prev_same_speaker` predictions. The main summary is written to `analysis/summary.json`.


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/analyze_enni_context_ablation.py \
  --utterance-only "$EXISTING_UTTERANCE_ONLY_JSONL" \
  --prev-same-speaker "$PREV_OUTPUT_DIR/predictions_prev_same_speaker.jsonl" \
  --prepared-input-jsonl "$FULL_INPUT_JSONL" \
  --out-dir "$PREV_OUTPUT_DIR/analysis"


In [ ]:
import json
from pathlib import Path

summary_path = Path(REPO_DIR) / PREV_OUTPUT_DIR / "analysis" / "summary.json"
print(summary_path)
print(summary_path.read_text(encoding="utf-8"))


This notebook is configured for the full reviewed ENNI run. If you need a smoke test, temporarily set `LIMIT` to a small number and use a different `PREV_OUTPUT_DIR`. Do not rerun `utterance_only` unless you intentionally want to replace the baseline file.
